# Egen Workshop: Evaluating a Staffing Agent

**Bench** is an internal staffing assistant. Delivery managers ask it things like *"Need a senior BigQuery engineer full-time from Jan 11"* and it replies with a shortlist.

The agent is already built. Your job today is to find out how good it is:

1. **Run** the agent over a reference dataset, scored by three evaluators we already know we want.
2. **Annotate** the replies by hand in Arize: pass/fail plus a short note on what went wrong.
3. **Group** your notes into error patterns.
4. **Write a judge** in Arize for a pattern the three evaluators don't catch.

Optional at the end: fix one of the problems you found and measure the change.

## Setup

In [ ]:
import sys
if "google.colab" in sys.modules:  # local venvs made with uv have no pip; uv is already installed there
    %pip install -q uv
!uv pip install -q openai==2.31.0 openai-agents==0.13.6 arize==8.64.0 arize-otel==0.12.0 openinference-instrumentation-openai-agents==1.4.1 pandas==2.3.3 python-dotenv==1.2.2 ipywidgets==8.1.9

In [ ]:
if "google.colab" in sys.modules:  # running from the repo locally? the code is already here
    !git clone -q https://github.com/levelup-labs-ai/egen-workshop.git

### Secrets

Open the **Secrets** panel (key icon on the left) and add these, with notebook access turned on:
- `ARIZE_API_KEY` and `ARIZE_SPACE_ID` (from the pre-read)
- `AZURE_OPENAI_ENDPOINT` and `AZURE_OPENAI_API_KEY` (we'll hand these out in the session)
- Or, to call OpenAI directly instead: `OPENAI_API_KEY`

Then pick the provider and the models below. The agent and the judge can use different models.

In [ ]:
PROVIDER = "azure"             # "azure" or "openai"
AGENT_MODEL = "gpt-5.6-terra"  # azure: "gpt-5.6-terra" or "gpt-5.4-mini"; openai: "gpt-6-luna"
JUDGE_MODEL = "gpt-5.6-terra"

import json
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_colwidth", None)

REPO = next(p for p in [Path.cwd(), Path.cwd() / "egen-workshop"] if (p / "workshop_helpers").exists())
sys.path.insert(0, str(REPO))

from workshop_helpers import tools
from workshop_helpers.setup import setup_clients
from workshop_helpers.evals import CASES, no_ineligible_pick, found_good_fit, make_grounded_judge
from workshop_helpers.experiments import ensure_dataset, run_experiment, reference_frame, score_summary, show_output

env = setup_clients(PROVIDER)
openai_client, arize, SPACE_ID = env["openai"], env["arize"], env["space_id"]
print(f"Connected to {PROVIDER}: agent {AGENT_MODEL}, judge {JUDGE_MODEL}. Traces go to Arize project:", env["project_name"])

---
## Part 1: The data

Bench reads from a small staffing database: consultants, projects, who is assigned where (and at what % of their time), and PTO.
Everything is made up, but it's shaped like a real resourcing system. **Today is Monday, Nov 16, 2026** in this world.

In [ ]:
consultants = pd.DataFrame(tools.DATA["consultants"])
print(len(consultants), "consultants")
consultants[["id", "name", "title", "level", "location", "skills", "industries"]].head(10)

In [ ]:
assignments = pd.DataFrame(tools.DATA["assignments"]).merge(
    pd.DataFrame(tools.DATA["projects"])[["id", "client", "name"]].rename(columns={"id": "project_id", "name": "project"}))
assignments[["consultant_id", "client", "project", "allocation_pct", "start", "end"]].head(10)

### The reference dataset

25 staffing requests, written the way managers actually post them in Slack. For each one, someone who knows the bench wrote down the reference answer:

- **good_fits**: people who would be a correct recommendation
- **must_not_recommend**: people who look right at first glance but aren't (booked, wrong timezone, expired certification...), with the reason
- **caveats**: things the manager must be told if a given person is recommended (still 50% on another project, PTO mid-engagement, certification expiring)

That's how most reference datasets get built: real inputs, plus a domain expert's answer. It takes a few hours, and it's the most useful few hours in the whole eval process.
We've done it for you today so we can spend the time on error analysis.

In [ ]:
reference = reference_frame()
reference[["case_id", "request", "good_fits", "must_not_recommend", "caveats"]].head(6)

In [ ]:
DATASET_ID = ensure_dataset(arize, SPACE_ID)
print("Arize dataset id:", DATASET_ID)

---
## Part 2: The agent

One agent, four tools. The tools are plain Python functions over the JSON data.

In [ ]:
for f in tools.TOOL_FUNCTIONS:
    print(f"{f.__name__}: {f.__doc__.strip().splitlines()[0]}")

This is the first-draft prompt. It's what most of us would write on day one.

In [ ]:
PROMPT_V1 = """
You are Bench, Fernhill's staffing assistant. Delivery managers and account leads ask you to find consultants for client work.
Today is Monday, November 16, 2026.

Use the tools to find consultants who match the request. Recommend up to 3 people, with a one-line reason for each.
Write a short message to the manager summarizing your recommendation.
"""

In [ ]:
demo = CASES["S01"]
print(demo["request"], "\n")
show_output(tools.run_agent(demo["request"], PROMPT_V1, AGENT_MODEL))

Open Arize → **Projects** → the project printed in setup. You'll see this run as a trace: the agent span, each LLM call, and each tool call with its inputs and outputs.

Compare the answer to the reference for S01:

In [ ]:
reference[reference.case_id == "S01"][["good_fits", "must_not_recommend", "caveats"]]

### Try to break Bench

Ask for someone the way you'd really ask on Slack: a role you staffed last month, a vague ask, a client you know.
Each request shows up as a trace in Arize. If you get a bad answer, remember what you typed. We'll come back to it.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

ask_box = widgets.Text(placeholder="e.g. Need a Kafka person at 50% for Crestline from January", layout=widgets.Layout(width="70%"))
ask_button = widgets.Button(description="Ask Bench", button_style="primary")
bench_reply = widgets.Output()

def ask_bench(_):
    with bench_reply:
        bench_reply.clear_output()
        print("Bench is looking...")
        out = tools.run_agent(ask_box.value, PROMPT_V1, AGENT_MODEL)
        bench_reply.clear_output()
        show_output(out)

ask_button.on_click(ask_bench)
display(widgets.HBox([ask_box, ask_button]), bench_reply)

---
## Part 3: The evaluators we start with

Before reading a single reply, there are things we already know we want to measure. Each one compares Bench's reply to the reference answer for that request.

| Evaluator | What it checks | Type |
|---|---|---|
| `no_ineligible_pick` | Nobody recommended is ruled out: booked too heavily for those dates, too junior, or missing a required certification | code |
| `found_good_fit` | At least one pick is on the reference list of good fits. When nobody fits, Bench should recommend no one | code |
| `grounded` | Everything the message says about a consultant is backed by what the tools returned | **LLM judge** |

Code checks are cheap, fast, and exact. Use code whenever you can.
`grounded` is about what the reply *says*, in whatever words the agent chose, so it needs an LLM to read it.

A judge works best when it's a narrow yes/no question and it's given the facts to check against. Here it gets the raw tool results.

In [ ]:
GROUNDED_JUDGE_PROMPT = """
You check a staffing assistant's reply for factual errors about consultants.
You get the request, the assistant's message, and the raw tool results the assistant saw.

Label "fail" only if the message states a fact that the tool results contradict or don't contain, such as:
- a name attached to the wrong consultant id
- a skill, certification, industry, location, or seniority the consultant doesn't have in the tool results
- availability that conflicts with the assignments or PTO in the tool results (e.g. "fully available" when they have PTO in the window)
Ignore opinions and adjectives ("strong", "great fit", "experienced"). Ignore people who are only mentioned as not available.
Label "pass" otherwise.

Return JSON: {"label": "pass" or "fail", "reason": "<one sentence; quote the wrong claim if any>"}
"""
grounded = make_grounded_judge(openai_client, GROUNDED_JUDGE_PROMPT, JUDGE_MODEL)
EVALUATORS = [no_ineligible_pick, found_good_fit, grounded]

---
## Part 4: Run the agent over the reference dataset

This runs Bench on all 25 requests as an Arize experiment and scores each reply with the three evaluators. It takes a few minutes.

In [ ]:
def task_v1(dataset_row) -> str:
    return json.dumps(tools.run_agent(dataset_row["request"], PROMPT_V1, AGENT_MODEL))

exp_v1, scored_v1 = run_experiment(arize, SPACE_ID, DATASET_ID, "bench-v1", task_v1, EVALUATORS)
print("Experiment:", exp_v1.name)
score_summary(scored_v1)

These numbers only cover what we thought to measure. A reply can pass all three and still be one you wouldn't act on. The next part is how you find out.

---
## Part 5: Error analysis (in Arize)

This is the most important part of the workshop. Read the replies yourself before trusting any score.

The three evaluators have already checked the facts: who is booked, who is senior enough, whether the message matches the data. Don't re-check them.
You sent this request and got this reply. **Could you act on it as it stands?** Fail it if:

- it answers a different question from the one you asked
- it filled in something you never said (a role, a skill, dates)
- it doesn't tell you who to pick, or why

1. In Arize, open **Datasets** → `bench-reference-requests` → **Experiment Analysis** tab → click the `bench-v1` experiment.
2. Click **Columns** (top right of the table) and tick `request` and `output.message`. Each row now shows what was asked and what Bench replied.
   To see how Bench got there, click the **trace icon** (top right of the output cell). The left side shows every LLM call and tool call; the right side shows the request and the reference answer (`good_fits`, `must_not_recommend`, `caveats`).
3. Then click the **annotate icon** next to it. The first time, set up the two fields you'll fill in. Click **Add / Remove Configs** → **Add New Annotation** and create:
   - `verdict`: type **Categorical Options**, with two labels: `fail` (score 0) and `pass` (score 1)
   - `note`: type **Freeform Text**

   Then tick both under **Add / Remove Configs** so they show in the panel.
   On the first row you annotate, fill in **both** fields, with a note even if the reply passes. A field only stays on the following rows once it has one saved value.
4. The annotation panel shows only the first lines of Bench's raw output, so read `request` and `output.message` in the table row first. Then, for each row, fill in:
   - **verdict**: `pass` or `fail`.
   - **note**: one sentence on the *first* thing that's wrong with the reply. Be specific: *"Lists three people but doesn't say which one to pick"*, not *"bad reply"*.
   Use **Next** to move to the next row. Annotations save automatically, and **Next** waits a second while they do.
5. Take about five minutes and annotate as many rows as you can. It's about 30 seconds a row, and you don't need all 25. Include rows that passed all three evaluators.

---
## Part 6: A judge for a pattern you found (in Arize)

Some of what you noted is already covered: a booked person being recommended shows up as a `no_ineligible_pick` failure.
The pattern that needs a new judge is the biggest one that **none of the three evaluators catches**.

Before you build it, write down:

- **The question**, as a narrow yes/no: does this reply have the pattern? One thing only. *"Is this a good reply?"* is too broad to judge.
- **What the judge needs to see** to answer it: the request, Bench's reply, and anything from the reference answer.
- **One reply that has the pattern and one that doesn't**, from the rows you annotated.

Then, on the experiment page in Arize:

1. Open **Alyx** (top right). Ask it to read your notes on the rows you marked fail, group them into patterns, and say which of the three evaluators already catches each. Compare its groups with your own impression, and pick the pattern. Give it a short name, for example `no_clear_pick`.
2. Ask Alyx to build an **LLM-as-a-judge evaluator** for your pattern. Nobody will read every reply by hand, so the judge is what checks for the pattern on this run and every run after it. The judge runs on a model set up in your Arize space; if Alyx can't find one, add it under **Settings** → **AI Providers** first. Name it after the pattern with `_judge` on the end. Give it two labels, `yes` and `no`, and say in plain words when the answer is `yes`. Read the template it drafts, then click **Accept**.
3. Click **Add Evaluator** (top right of the experiment page) and pick your evaluator under **Evaluator Hub**.
4. Under *Pick the examples to evaluate*, change **Dataset** to **Experiment** and select `bench-v1`.
5. Under *Connect the eval to your examples*, map `{request}` to `request` and the output variable to `output`. Click **Run Evaluator(s)**. The scores show up on each row next to the other three.
6. Now label the pattern yourself. The judge gives every row a `yes` or a `no`, and you don't know yet whether to believe it. Your `verdict` won't do for this: it says whether a reply is usable, not why. Click the **annotate icon** → **Add / Remove Configs** → **Add New Annotation**, use the pattern's name (the judge's name without `_judge`) and keep **Categorical Options**. Under **Optimization Direction** choose **Minimize**, because a `yes` is the bad outcome. The two label rows swap when you do: type `yes` in the red row (score 1) and `no` in the green row (score 0). Click **Create**, then mark the rows `yes` or `no`, aiming for all 25. Label from your own notes, before you look at what the judge said on that row. Rows you passed are almost always `no`. A row you failed for a different reason is also `no`.

Then check alignment, meaning how often the judge agrees with you. Row by row, compare the judge's `yes` or `no` with the label you gave in step 6.
Don't compare it with `verdict`: a reply can fail for a different reason, and the judge is right to say `no` on it.
Where the two differ, read the judge's reason. Sometimes the judge is wrong (fix its prompt). Sometimes you missed something.
The judge is usually off on the first try. If there's time, fix it: tell Alyx which rows differ and why, ask it to update the judge's prompt, and click **Accept**. Then run the judge again yourself: click **Evaluators** at the top of the experiment page, then **Run** next to your judge. Open **Advanced** and choose **Override evaluation labels**, or the run skips every row that already has a label. Then check again. One or two rounds is normal, and it doesn't have to reach 100%.
This is the step people skip. Before you use a judge to improve Bench, make sure the judge is right.

---
## Optional: Fix the prompt and measure it

Rewrite the prompt to address a pattern you found, run it as `bench-v2` with the same evaluators, and compare.
Change one or two things at a time, so you know what moved the numbers. Run it twice before you believe a small change.

In [ ]:
PROMPT_V2 = PROMPT_V1  # edit me

def task_v2(dataset_row) -> str:
    return json.dumps(tools.run_agent(dataset_row["request"], PROMPT_V2, AGENT_MODEL))

exp_v2, scored_v2 = run_experiment(arize, SPACE_ID, DATASET_ID, "bench-v2", task_v2, EVALUATORS)
pd.concat({"v1": score_summary(scored_v1).set_index("metric"), "v2": score_summary(scored_v2).set_index("metric")}, axis=1)